In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/lurenhaothu/SPW
%cd SPW
!cp -v /content/drive/MyDrive/CWMI_data/data.zip /content/SPW/
!unzip -oq data.zip

Mounted at /content/drive
/content
Cloning into 'SPW'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 130 (delta 60), reused 94 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 7.55 MiB | 15.94 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/SPW
'/content/drive/MyDrive/CWMI_data/data.zip' -> '/content/SPW/data.zip'


In [ ]:
%cd /content/SPW
!python eval_qualitative.py

/content/SPW
fold: 0
fold: 1
fold: 2
finished  U_Net_SNEMI3D_SPWH_loss_N4_K4_b0.9_l10_p_2025-03-07-02-32-56  with time:  20.056339025497437
fold: 0
fold: 1
fold: 2
finished  U_Net_SNEMI3D_SPWH_loss_N4_K4_b0.9_l10_p_2025-03-07-02-41-54  with time:  19.458553314208984
fold: 0
fold: 1
fold: 2
finished  U_Net_SNEMI3D_SPWH_loss_N4_K4_b0.9_l10_p_2025-03-07-02-50-51  with time:  19.42789101600647


In [ ]:
%cd /content/SPW
import torch
from sklearn.model_selection import KFold
from model.dataset import Dataset
from torch.utils.data import DataLoader
from model.unet import UNet
from PIL import Image
import numpy as np
import model.metrics as metrics
import os
import pandas as pd
import matplotlib.pyplot as plt
import random
import model.loss as loss_functions
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from data.map_gen_ABW import WeightMapLoss
from model.clDice.clDice import soft_dice_cldice
from model.rmi.rmi import RMILoss
# from model.CWMI_loss.CWMI_loss import CWMI_loss
from model.CWMI_loss.CWMI_loss import CWMI_loss
from model.SPW_loss.SPW_loss import SPW_loss
from model.SPW_loss.SW_loss import SW_loss
from plot_result import plot_result
from model.skea_topo.skea_topo_loss import Skea_topo_loss
import model.att_unet as models
import psutil

# from VM_UNet.models.vmunet.vmunet import VMUNet

lossFuncs = [(loss_functions.BCELoss(), ""),
             #(loss_functions.BCE_withClassBalance(), ""),
             #(loss_functions.DiceLoss(), ""),
             #(loss_functions.FocalLoss(gamma=2.5), ""),
             #(loss_functions.JaccardLoss(), ""),
             #(loss_functions.TverskyLoss(alpha=0.5, beta=0.5), ""),
             #(loss_functions.Unet_Weight_BCE(), ""),
             #(WeightMapLoss(), ""),
             #(RMILoss(num_classes=2), ""),
             #(soft_dice_cldice(), ""),
             #(Skea_topo_loss(), ""),
             #(CWMI_loss(complex=True, spN=4, spK=4, lamb=0.9), "CWMI_optimal"),
             #(SPW_loss(spN=4, spK=4, beta=0.9, lamb=10, base_loss='bce'), \
             #  "N4_K4_b0.9_l10_summap_bce"),
             #(SPW_loss(spN=4, spK=6, beta=0.9, lamb=10, base_loss='bce'), \
             #  "N4_K6_b0.9_l10_summap_bce"),
             #(SPW_loss(spN=4, spK=8, beta=0.9, lamb=10, base_loss='bce'), \
             #  "N4_K8_b0.9_l10_summap_bce"),
             #(SensitiveLoss(), "")
             #(SPW_loss(spN=5, spK=8, beta=0.9, lamb=10, base_loss='bce'), \
             #   "N5_K8_b0.9_l10_summap_bce"),
             #(SPW_loss(spN=6, spK=8, beta=0.9, lamb=10, base_loss='bce'), \
             #   "N6_K8_b0.9_l10_summap_bce"),
             ]

model_classes = [ models.U_Net,
                 #models.AttU_Net,
                 # VMUNet
                 ]

# dataset_names = ["DRIVE", "GlaS", "mass_road", "SNEMI3D"]
dataset_names = ["SNEMI3D"]

def experiment(dataset_name, model_class, lossFunc, note=''):

    val_metric = metrics.miou

    test_metrics = [metrics.miou, metrics.vi, metrics.mdice, metrics.ari, metrics.hausdorff_distance]

    batch_size = 10

    epoch_num = 50

    gradient_plt = False

    expriment_name = model_class.__name__ + "_" + dataset_name + "_" + lossFunc.__class__.__name__ + "_" + note + "_" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

    cwd = os.getcwd()
    curResultDir = cwd + "/results/" + expriment_name + "/"
    os.makedirs(curResultDir, exist_ok=True)

    load_pretrain = False
    pretrain_dir_name = "SNEMI3D_BCE_CLassW_val_miou_btchSize_10_2025-01-14-18-43-31"

    pretrain_dir = cwd + "/results/" + pretrain_dir_name + "/"

    numFile = len(os.listdir(cwd + "/data/" + dataset_name + "/masks/"))
    fileList = [i for i in range(numFile)]

    fold_num = 3
    kf = KFold(n_splits=fold_num, shuffle=True)

    for fold, (train_and_val_list, test_list) in enumerate(kf.split(fileList)):

        print("fold: " + str(fold))

        train_size = int(0.8 * len(train_and_val_list))
        val_size = len(train_and_val_list) - train_size

        train_list = random.sample(train_and_val_list.tolist(), train_size)
        val_list = [i for i in train_and_val_list if i not in train_list]

        df = pd.DataFrame({'TrainNumbers': train_list})
        df.to_csv(curResultDir + 'Fold_' + str(fold) + '_train.csv', index=False)
        df = pd.DataFrame({'ValNumbers': val_list})
        df.to_csv(curResultDir + 'Fold_' + str(fold) + '_val.csv', index=False)
        df = pd.DataFrame({'TestNumbers': test_list.tolist()})
        df.to_csv(curResultDir + 'Fold_' + str(fold) + '_test.csv', index=False)

        if lossFunc.__class__.__name__ == "Unet_Weight_BCE":
            weight_map_name = "unet_maps"
        elif lossFunc.__class__.__name__ == "WeightMapLoss":
            weight_map_name = "ABW_maps"
        elif lossFunc.__class__.__name__ == "Skea_topo_loss":
            weight_map_name = "skea_topo_maps"
        else:
            weight_map_name = None

        train_dataset = Dataset(dataset_name, train_list, augmentation=True, weight_map_name=weight_map_name)
        val_dataset = Dataset(dataset_name, val_list, augmentation=False, weight_map_name=weight_map_name)
        test_dataset = Dataset(dataset_name, test_list, augmentation=False)

        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)
        test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

        if model_class.__name__ != "VMUNet":
            if dataset_name == "SNEMI3D":
                model = model_class(img_ch=1, output_ch=2)
            else: # all other datasets are RGB
                model = model_class(img_ch=3, output_ch=2)
        else:
            if dataset_name == "SNEMI3D":
                model = VMUNet(
                    num_classes=2,
                    input_channels=1,
                    drop_path_rate=0
                )
            else:
                model = VMUNet(
                    num_classes=2,
                    input_channels=3,
                    drop_path_rate=0
                )

        if load_pretrain:
            prev_dict = torch.load(pretrain_dir + "fold_" + str(fold) + "_best_model_state.pth")
            model.load_state_dict(prev_dict)

        model.cuda()

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)

        val_res_list = []
        VRAM_RAM_list = []
        curmin_val_res = None

        for epoch in range(epoch_num):
            print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
            print("epoch: " + str(epoch))

            t1 = time.time()

            model.train()
            for i, (image, mask, w_map, class_weight) in enumerate(train_dataloader):
                image = image.cuda()
                mask = mask.cuda()
                w_map = w_map.cuda()
                torch.cuda.reset_peak_memory_stats()
                pred = torch.softmax(model(image), 1)[:, 1:2, :, :]
                # print(pred.shape, mask.shape, w_map.shape)
                vram_1 = torch.cuda.memory_allocated()
                loss = lossFunc(mask, pred, w_map, class_weight.cuda(), epoch)
                vram_2 = torch.cuda.memory_allocated()
                     # + loss_functions.DiceLoss()(mask, pred, w_map, class_weight.cuda(), epoch)
                if i == 0:
                  print(loss)

                optimizer.zero_grad()
                loss.backward()
                vram_3 = torch.cuda.memory_allocated()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
                optimizer.step()
                peak = torch.cuda.max_memory_allocated()

                #print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")
                #print(f"Reserved: {torch.cuda.memory_reserved() / 1e6:.2f} MB")
                VRAM_RAM_list.append([vram_1, vram_2, vram_3, peak])

            scheduler.step()

            t2 = time.time()

            print("train time: ", t2 - t1)

            # torch.save(unet.state_dict(), "epoch_" + str(i) + ".pth")

            # validation
            images = []
            masks = []
            preds = []
            preds_bin = []
            for index, (image, mask, w_map, class_weight) in enumerate(val_dataloader):
                model.eval()
                with torch.no_grad():
                    pred = torch.softmax(model(image.cuda()), 1)[:, 1:2, :, :]
                    images.append(image.squeeze().numpy())
                    masks.append(mask.squeeze().numpy())
                    preds.append(pred.cpu().squeeze().numpy())
                    preds_bin.append((preds[-1] > 0.5).astype(int))

                # TODO: calculate loss grad on pixels

                if gradient_plt and epoch % 10 == 9  and index == 0:
                    plot_figures = []
                    plot_figures.append(image.squeeze().numpy())
                    plot_figures.append(mask.squeeze().numpy())
                    plot_figures.append(pred.cpu().squeeze().numpy())
                    plot_figures.append((pred.cpu().squeeze().numpy() > 0.5).astype(int))
                    dif = mask.squeeze().numpy() - (pred.cpu().squeeze().numpy() > 0.5).astype(int)
                    plot_figures.append(dif)

                    pred.requires_grad = True
                    l = lossFunc(mask.cuda(), pred, w_map.cuda(), class_weight.cuda())
                    grad = torch.autograd.grad(l, pred)
                    plot_figures.append(grad[0].cpu().squeeze().numpy())

                    fig_path = curResultDir + "fold_" + str(fold) + "_epoch_" + str(epoch) + "_val_sample.png"
                    plot_result(plot_figures, fig_path, size=plot_figures[-1].shape)


            with ThreadPoolExecutor(max_workers=10) as executor:
                val_res_temp = list(executor.map(val_metric, preds_bin, masks))
            val_res = np.mean(val_res_temp)

            t3 = time.time()
            print("val time: ", t3 - t2)

            val_res_list.append(val_res)
            result = pd.DataFrame({
                "Epoch": [epoch],
                val_metric.__name__: [val_res],
            })
            if not os.path.exists(curResultDir + "fold_" + str(fold) + "_val_result.csv"):
                result.to_csv(curResultDir + "fold_" + str(fold) + "_val_result.csv", index=False)
            else:
                result.to_csv(curResultDir + "fold_" + str(fold) + "_val_result.csv", mode='a', header=False, index=False)

            VRAM_RAM_frame = pd.DataFrame({
                "Epoch": [epoch],
                "Before forward": [VRAM_RAM_list[-1][0]],
                "After forward": [VRAM_RAM_list[-1][1]],
                "After backprop": [VRAM_RAM_list[-1][2]],
                "Peak VRAM": [VRAM_RAM_list[-1][3]]
            })
            if not os.path.exists(curResultDir + "fold_" + str(fold) + "_VRAM_RAM_usage.csv"):
                VRAM_RAM_frame.to_csv(curResultDir + "fold_" + str(fold) + "_VRAM_RAM_usage.csv", index=False)
            else:
                VRAM_RAM_frame.to_csv(curResultDir + "fold_" + str(fold) + "_VRAM_RAM_usage.csv", mode='a', header=False, index=False)

            if curmin_val_res == None or val_res < curmin_val_res:
                curmin_val_res = val_res
                torch.save(model.state_dict(), curResultDir + "fold_" + str(fold) + "_best_model_state.pth")
                print("save best model")

            print("epoch: " + str(epoch) + " " + val_metric.__name__ + ": " + str(val_res))

        #test
        best_state_dict = torch.load(curResultDir + "fold_" + str(fold) + "_best_model_state.pth", weights_only=True)
        model.load_state_dict(best_state_dict)
        model.eval()

        images_test = []
        masks_test = []
        preds_test = []
        preds_bin_test = []
        test_results = []
        for index, (image, mask, _, _) in enumerate(test_dataloader):
            model.eval()
            with torch.no_grad():
                pred = torch.softmax(model(image.cuda()), 1)[:, 1:2, :, :]
                images_test.append(image.squeeze().numpy())
                masks_test.append(mask.squeeze().numpy())
                preds_test.append(pred.cpu().squeeze().numpy())
                preds_bin_test.append((preds_test[-1] > 0.5).astype(int))

        for test_metric in test_metrics:
            with ThreadPoolExecutor(max_workers=10) as executor:
                test_result = list(executor.map(test_metric, preds_bin_test, masks_test))
            test_results.append((test_metric.__name__, np.mean(test_result)))

        test_result_pd = {"Fold": [fold]}
        for metric_name, value in test_results:
            test_result_pd[metric_name] = [value]

        test_result_pd = pd.DataFrame(test_result_pd)

        if not os.path.exists(curResultDir + "_test_result.csv"):
            test_result_pd.to_csv(curResultDir + "_test_result.csv", index=False)
        else:
            test_result_pd.to_csv(curResultDir + "_test_result.csv", mode='a', header=False, index=False)

        print("---------------------------------------------------------")
        print(expriment_name)
        print("---------------------fold " + str(fold) + " finished---------------------")
        for metric_name, value in test_results:
            print(metric_name + ": " + str(value))
        print("---------------------------------------------------------")
        torch.cuda.empty_cache()

for  model_class in model_classes:
    for lossFunc, note in lossFuncs:
        for dataset_name in dataset_names:
            experiment(dataset_name, model_class, lossFunc, note)

/content/SPW
fold: 0
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
epoch: 0
tensor(0.6253, device='cuda:0', grad_fn=<MeanBackward0>)
train time:  9.748128890991211
val time:  2.857517957687378
save best model
epoch: 0 miou: -0.41525108348234513
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
epoch: 1
tensor(0.5139, device='cuda:0', grad_fn=<MeanBackward0>)
train time:  7.411947965621948
val time:  2.857862949371338
save best model
epoch: 1 miou: -0.471568346265685
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
epoch: 2
tensor(0.4718, device='cuda:0', grad_fn=<MeanBackward0>)
train time:  7.4602978229522705
val time:  2.8823890686035156
save best model
epoch: 2 miou: -0.4827600803404963
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
epoch: 3
tensor(0.4204, device='cuda:0', grad_fn=<MeanBackward0>)
train time:  7.469233989715576
val time:  2.878631114959717
save best model
epoch: 3 miou: -0.5023384517379229
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
epoch: 4
tensor(0.3724, device='cuda:0', grad_fn=<MeanBackward0>)
train time:  

In [ ]:
!cp -vr /content/SPW/results/ /content/drive/MyDrive/SPW_data/rebuttal/spK/
#from google.colab import runtime
#runtime.unassign()

'/content/SPW/results/' -> '/content/drive/MyDrive/SPW_data/rebuttal/spK/'
'/content/SPW/results/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32' -> '/content/drive/MyDrive/SPW_data/rebuttal/spK/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32'
'/content/SPW/results/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32/Fold_0_train.csv' -> '/content/drive/MyDrive/SPW_data/rebuttal/spK/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32/Fold_0_train.csv'
'/content/SPW/results/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32/Fold_0_val.csv' -> '/content/drive/MyDrive/SPW_data/rebuttal/spK/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32/Fold_0_val.csv'
'/content/SPW/results/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32/Fold_0_test.csv' -> '/content/drive/MyDrive/SPW_data/rebuttal/spK/U_Net_SNEMI3D_SPW_loss_N4_K4_b0.9_l10_summap_bce_2025-05-16-20-43-32/Fo

In [ ]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip uninstall mamba-ssm causal-conv1d
!pip install causal-conv1d && pip install mamba-ssm==2.2.2

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.1/799.1 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 57.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 103.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 125.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 44.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

In [ ]:
#%cd /content/SPW
#!git clone https://github.com/JCruan519/VM-UNet.git
# !pip install mamba_ssm==2.2.4 # monai causal-conv1d

# Copyright (c) OpenMMLab. All rights reserved.
import functools

import numpy as np
import torch
import torch.nn.functional as F
#from mmengine.fileio import load

def reduce_loss(loss, reduction) -> torch.Tensor:
    """Reduce loss as specified.

    Args:
        loss (Tensor): Elementwise loss tensor.
        reduction (str): Options are "none", "mean" and "sum".

    Return:
        Tensor: Reduced loss tensor.
    """
    reduction_enum = F._Reduction.get_enum(reduction)
    # none: 0, elementwise_mean:1, sum: 2
    if reduction_enum == 0:
        return loss
    elif reduction_enum == 1:
        return loss.mean()
    elif reduction_enum == 2:
        return loss.sum()


def weight_reduce_loss(loss,
                       weight=None,
                       reduction='mean',
                       avg_factor=None) -> torch.Tensor:
    """Apply element-wise weight and reduce loss.

    Args:
        loss (Tensor): Element-wise loss.
        weight (Tensor): Element-wise weights.
        reduction (str): Same as built-in losses of PyTorch.
        avg_factor (float): Average factor when computing the mean of losses.

    Returns:
        Tensor: Processed loss values.
    """
    # if weight is specified, apply element-wise weight
    if weight is not None:
        assert weight.dim() == loss.dim()
        if weight.dim() > 1:
            assert weight.size(1) == 1 or weight.size(1) == loss.size(1)
        loss = loss * weight

    # if avg_factor is not specified, just reduce the loss
    if avg_factor is None:
        loss = reduce_loss(loss, reduction)
    else:
        # if reduction is mean, then average the loss by avg_factor
        if reduction == 'mean':
            # Avoid causing ZeroDivisionError when avg_factor is 0.0,
            # i.e., all labels of an image belong to ignore index.
            eps = torch.finfo(torch.float32).eps
            loss = loss.sum() / (avg_factor + eps)
        # if reduction is 'none', then do nothing, otherwise raise an error
        elif reduction != 'none':
            raise ValueError('avg_factor can not be used with reduction="sum"')
    return loss


def weighted_loss(loss_func):
    """Create a weighted version of a given loss function.

    To use this decorator, the loss function must have the signature like
    `loss_func(pred, target, **kwargs)`. The function only needs to compute
    element-wise loss without any reduction. This decorator will add weight
    and reduction arguments to the function. The decorated function will have
    the signature like `loss_func(pred, target, weight=None, reduction='mean',
    avg_factor=None, **kwargs)`.
    """

    @functools.wraps(loss_func)
    def wrapper(pred,
                target,
                weight=None,
                reduction='mean',
                avg_factor=None,
                **kwargs):
        # get element-wise loss
        loss = loss_func(pred, target, **kwargs)
        loss = weight_reduce_loss(loss, weight, reduction, avg_factor)
        return loss

    return wrapper

import torch.nn.functional as F
import math
import torch.nn as nn

@torch.no_grad()
def sensitivity(pred, gamma=1.0, scale=False):
    n_class = pred.size(1)
    eps = torch.finfo(pred.dtype).eps
    score = F.softmax(pred, dim=1).clamp(min=eps, max=1. - eps)
    min_real = torch.finfo(pred.dtype).min
    log_score = F.log_softmax(pred, dim=1).clamp(min=min_real)
    entropy = -torch.sum(score * log_score, dim=1)
    sen = torch.pow(entropy / math.log(n_class), exponent=gamma)
    # logger.info(round(sen.mean().cpu().item(), 5))  # for plot
    if scale:
        sen = sen / (sen.mean() + eps)  # approximate the lr policy
    # print(sen)
    return sen

class SensitiveLoss(nn.Module):
    def __init__(self, gamma=1.0, use_scale=False, reduction='mean', class_weight=None,
                 loss_weight=1.0, loss_name='loss_sensitive', **kwargs):
        super(SensitiveLoss, self).__init__()
        self.gamma = gamma
        self.use_scale = use_scale
        self.reduction = reduction
        self.loss_weight = loss_weight
        self._loss_name = loss_name

    # def forward(self, pred, target, reduction_override=None, scale_override=None, ignore_index=255, **kwargs):
    def forward(self, mask, pred, w_map, class_weight, epoch=None):

        pred = torch.concat((pred, 1-pred), dim=1)
        mask = torch.concat((mask, 1-mask), dim=1)

        reduction = self.reduction
        scale = self.use_scale
        sen = sensitivity(pred, gamma=self.gamma, scale=scale)
        loss = F.cross_entropy(pred, mask, weight=None, reduction='none')
        loss = weight_reduce_loss(loss, weight=sen, reduction=reduction)
        return loss * self.loss_weight